In [ ]:
# Install the necessary packages
# langchain_community : loaders, tools, and community integrations
# tiktoken            : OpenAI tokenizer used by RecursiveCharacterTextSplitter
# langchain-openai    : OpenAI wrappers (kept for tiktoken compatibility)
# langchainhub        : pull prompts from the LangChain Hub
# chromadb            : local vector store
# langchain           : core orchestration framework
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
# google-genai : official Google Generative AI SDK (required by langchain-google-genai)
! pip install google_genai

In [ ]:
# langchain-google-genai : LangChain wrappers for Gemini LLM and Gemini Embeddings
! pip install langchain-google-genai

In [ ]:
# Core imports
from langchain_core.prompts import ChatPromptTemplate          # Build structured prompts
from langchain_google_genai import GoogleGenerativeAIEmbeddings # Gemini embedding model
from langchain_google_genai import ChatGoogleGenerativeAI       # Gemini chat model
from langchain_core.output_parsers import StrOutputParser       # Parse LLM output to plain string
from langchain_community.document_loaders import UnstructuredHTMLLoader  # Load HTML files
from langchain_core.runnables import RunnablePassthrough        # Pass input through unchanged
from langchain_text_splitters import RecursiveCharacterTextSplitter      # Chunk documents
from langchain_chroma import Chroma                             # Local vector store

In [ ]:
import os

# ── LangSmith tracing (optional) ───────────────────────────────────────────
# Set LANGCHAIN_TRACING_V2=true to stream all LangChain runs to LangSmith.
# Useful for debugging chain steps and inspecting retrieved documents.
# Get a free key at https://smith.langchain.com
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = "https://api.smith.langchain.com"
os.environ['LANGCHAIN_API_KEY'] = "YOUR_LANGSMITH_API_KEY"   # <-- replace

# ── Google Gemini API key ───────────────────────────────────────────────────
# Free key available at https://aistudio.google.com  (no credit card needed)
API = "YOUR_GOOGLE_API_KEY"   # <-- replace

# Path to the HTML car manual (update to your local path)
sources = r"data\mg-zs-warning-messages.html"  # <-- replace

# ── LLM : Gemini 2.5 Flash ─────────────────────────────────────────────────
# temperature=0 → deterministic answers, better suited for factual RAG
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=API, temperature=0)

# ── Embedding model : Gemini Embedding 001 ─────────────────────────────────
# Converts text chunks and queries into 768-dimensional vectors for similarity search
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=API)

In [ ]:
# ── INDEXING : Step 1 — Load ────────────────────────────────────────────────
# UnstructuredHTMLLoader parses the HTML file and returns a list of LangChain Documents.
# Each Document has .page_content (text) and .metadata (source path, etc.)
loader = UnstructuredHTMLLoader(file_path=sources)
car_docs = loader.load()

In [ ]:
# ── INDEXING : Step 2 — Split ───────────────────────────────────────────────
# RecursiveCharacterTextSplitter splits text into overlapping chunks.
# from_tiktoken_encoder uses the GPT tokenizer to count tokens accurately.
#
# chunk_size=200   : each chunk contains at most 200 tokens
# chunk_overlap=50 : 50-token overlap between consecutive chunks
#                    ensures context is not lost at chunk boundaries
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200,
    chunk_overlap=50
)
splits = text_splitter.split_documents(car_docs)

In [ ]:
# ── INDEXING : Step 3 — Embed & Store ──────────────────────────────────────
# Chroma.from_documents embeds every chunk with the Gemini embedding model
# and stores the resulting vectors in a local in-memory Chroma collection.
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

In [ ]:
# ── RETRIEVAL : Create retriever ────────────────────────────────────────────
# as_retriever() wraps the vector store as a LangChain Retriever.
# k=2 : return the 2 most similar chunks for each query
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
# ── GENERATION : RAG prompt template ───────────────────────────────────────
# The LLM is instructed to answer ONLY from the retrieved context.
# This prevents hallucinations by grounding the model in the source document.
template = """Tu es un assistant expert. Réponds à la question en te basant uniquement sur le contexte fourni.
Si la réponse n'est pas dans le contexte, dis-le clairement.

Contexte : {context}

Question : {question}
"""

In [ ]:
# Context and question generation — placeholder cell
# (RAG Fusion query generation is defined in the next cells)

In [ ]:
# ── RAG FUSION : Query expansion prompt ────────────────────────────────────
# RAG Fusion improves retrieval by generating multiple paraphrased versions
# of the original question. Each variant targets different phrasings that
# may better match chunks in the vector store.
template1 = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template1)

In [ ]:
# ── RAG FUSION : Query generation chain ────────────────────────────────────
# Pipeline:
#   1. prompt_rag_fusion : formats the question into the expansion prompt
#   2. llm               : generates 4 related queries as a single string
#   3. StrOutputParser() : extracts the plain text from the LLM response
#   4. lambda            : splits by newline and strips whitespace;
#                          empty lines are filtered out to avoid embedding errors
generate_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
)

In [ ]:
from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """
    Merge and re-rank multiple ranked document lists using Reciprocal Rank Fusion (RRF).

    RRF assigns a score of 1 / (rank + k) to each document at each position,
    then sums scores across all lists. Documents appearing consistently at the
    top of multiple lists receive the highest combined scores.

    Time complexity: O(N * M * log(N * M)) where N = number of lists, M = docs per list.

    Args:
        results (list[list]): A list of ranked document lists, one per generated query.
        k (int): Smoothing constant that controls the influence of top-ranked documents.
                 Higher k flattens score differences between ranks. Default: 60.

    Returns:
        list[tuple]: Documents sorted by descending RRF score, as (Document, score) tuples.
    """
    # Dictionary mapping serialized document → cumulative RRF score
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            # Serialize the Document object to a JSON string to use as a hashable key
            doc_str = dumps(doc)

            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            previous_score = fused_scores[doc_str]
            # Core RRF formula: reward documents ranked early across multiple lists
            fused_scores[doc_str] += 1 / (rank + k)

    # Deserialize keys back to Document objects and sort by score (highest first)
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    return reranked_results


# Full RAG Fusion retrieval chain:
#   generate_queries          : expands the question into 4 variants
#   retriever.map()           : runs each variant through the vector store in parallel
#   reciprocal_rank_fusion    : merges and re-ranks all returned documents
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion

In [ ]:
# ── FINAL RAG CHAIN ─────────────────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from operator import itemgetter  # Extracts a key from the input dict inside a chain

prompt = ChatPromptTemplate.from_template(template)

# Full end-to-end chain:
#   Step 1 — Build context and question in parallel:
#       "context"  : runs the full RAG Fusion retrieval chain on the input question
#       "question" : extracts the raw question string from the input dict
#   Step 2 — Format both into the RAG prompt
#   Step 3 — Send the filled prompt to the Gemini LLM
#   Step 4 — Parse the LLM response to a plain Python string
final_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

question = "What are the warnings to care about"
final_rag_chain.invoke({"question": question})